In [54]:
# set the project root so notebook imports work from the notebooks folder
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_ROOT = PROJECT_ROOT / "Data"

print(PROJECT_ROOT)
print(DATA_ROOT)
print(DATA_ROOT.exists())

/Users/quentin/Documents/Computer/School/CMU/Spring_2026/38616/Gated-Risk-MCP
/Users/quentin/Documents/Computer/School/CMU/Spring_2026/38616/Gated-Risk-MCP/Data
True


In [55]:
# import the SROIE loader and feature/label builders from the repo code
import importlib
import pandas as pd

from utils.data_utils import load_sroie_split, preview_record
import src.sroie_features as sroie_features

importlib.reload(sroie_features)

<module 'src.sroie_features' from '/Users/quentin/Documents/Computer/School/CMU/Spring_2026/38616/Gated-Risk-MCP/src/sroie_features.py'>

In [56]:
# load the SROIE train split and confirm the raw pipeline is alive
records = load_sroie_split("train", data_root=DATA_ROOT)

print(len(records))
preview_record(records[0])

712
doc_id: X00016469612
dataset: SROIE
split: train
image_path: /Users/quentin/Documents/Computer/School/CMU/Spring_2026/38616/Gated-Risk-MCP/Data/SROIE/0325updated.task1train(626p)/X00016469612.jpg
image object present: False
image_size: None
num_tokens: 44
num_boxes: 44
num_fields: 4
field keys: ['company', 'date', 'address', 'total']
metadata keys: []


In [57]:
# build the receipt-level SROIE feature table from the loaded records
sroie_df = sroie_features.sroie_feature_dataframe(records)

display(sroie_df.head())
print(sroie_df.shape)
print(sorted(sroie_df.columns))

,doc_id,dataset,split,img_width,img_height,n_tokens,n_boxes,ocr_char_count,ocr_word_count,company_present,...,amount_token_ratio,date_token_ratio,avg_token_len,avg_words_per_token,anchors_present_count,fields_present_count,all_fields_present,any_field_missing,ocr_is_empty,aspect_ratio
0,X00016469612,SROIE,train,463.0,1013.0,44,44,441,84,True,...,0.159091,0.000000,10.022727,1.909091,3,4,True,False,False,0.457058
1,X00016469619,SROIE,train,439.0,1004.0,48,48,637,102,True,...,0.229167,0.000000,13.270833,2.125000,2,4,True,False,False,0.437251
2,X00016469620,SROIE,train,459.0,949.0,54,54,668,121,True,...,0.222222,0.000000,12.370370,2.240741,2,4,True,False,False,0.483667
3,X00016469622,SROIE,train,461.0,933.0,60,60,525,105,True,...,0.266667,0.016667,8.750000,1.750000,3,4,True,False,False,0.494105
4,X00016469623,SROIE,train,463.0,1026.0,61,61,735,142,True,...,0.245902,0.000000,12.049180,2.327869,2,4,True,False,False,0.451267


(712, 38)
['address_in_ocr', 'address_len', 'address_present', 'all_fields_present', 'amount_token_ratio', 'anchors_present_count', 'any_field_missing', 'aspect_ratio', 'avg_token_len', 'avg_words_per_token', 'company_in_ocr', 'company_len', 'company_present', 'dataset', 'date_in_ocr', 'date_len', 'date_present', 'date_token_ratio', 'doc_id', 'exact_total_matches', 'fields_present_count', 'has_cash_anchor', 'has_date_anchor', 'has_total_anchor', 'img_height', 'img_width', 'n_amount_like_tokens', 'n_boxes', 'n_date_like_tokens', 'n_tokens', 'ocr_char_count', 'ocr_is_empty', 'ocr_word_count', 'split', 'token_box_ratio', 'total_in_ocr', 'total_len', 'total_present']


In [58]:
# sanity-check the main numeric feature groups for missingness and scale
display(
    sroie_df[
        [
            "img_width",
            "img_height",
            "aspect_ratio",
            "n_tokens",
            "n_boxes",
            "ocr_char_count",
            "ocr_word_count",
            "company_len",
            "date_len",
            "address_len",
            "total_len",
            "exact_total_matches",
            "n_amount_like_tokens",
            "n_date_like_tokens",
            "token_box_ratio",
            "amount_token_ratio",
            "date_token_ratio",
            "avg_token_len",
            "avg_words_per_token",
            "anchors_present_count",
            "fields_present_count",
        ]
    ].describe()
)

,img_width,img_height,aspect_ratio,n_tokens,n_boxes,ocr_char_count,ocr_word_count,company_len,date_len,address_len,...,exact_total_matches,n_amount_like_tokens,n_date_like_tokens,token_box_ratio,amount_token_ratio,date_token_ratio,avg_token_len,avg_words_per_token,anchors_present_count,fields_present_count
count,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000,...,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000,712.000000
mean,1269.889045,2283.568820,0.511129,52.744382,52.744382,605.705056,111.411517,23.209270,9.004213,66.592697,...,2.127809,13.224719,0.196629,0.988764,0.229278,0.004237,11.768749,2.142303,2.497191,3.755618
std,1345.629413,1753.298861,0.118331,18.362251,18.362251,172.144405,33.476621,9.203269,2.469216,24.076177,...,1.486635,9.525895,0.447641,0.105477,0.095327,0.009749,2.612188,0.418657,0.576124,0.954284
min,436.000000,605.000000,0.263498,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,623.000000,1402.000000,0.422162,41.000000,41.000000,502.000000,89.750000,18.000000,8.000000,54.000000,...,1.000000,7.000000,0.000000,1.000000,0.160000,0.000000,9.878686,1.877841,2.000000,4.000000
50%,793.000000,1675.000000,0.497784,49.000000,49.000000,580.000000,106.000000,22.000000,10.000000,70.000000,...,2.000000,11.000000,0.000000,1.000000,0.225000,0.000000,11.716063,2.152681,3.000000,4.000000
75%,934.000000,2086.750000,0.577567,62.000000,62.000000,707.000000,128.000000,30.000000,10.000000,84.000000,...,3.000000,17.000000,0.000000,1.000000,0.281360,0.000000,13.861035,2.385052,3.000000,4.000000
max,4961.000000,7016.000000,0.971354,153.000000,153.000000,1270.000000,237.000000,49.000000,12.000000,135.000000,...,7.000000,76.000000,4.000000,1.000000,0.539007,0.056604,19.727273,3.545455,3.000000,4.000000


In [59]:
# inspect boolean feature rates to confirm recoverability and anchor signals look sensible
display(
    sroie_df[
        [
            "company_present",
            "date_present",
            "address_present",
            "total_present",
            "company_in_ocr",
            "date_in_ocr",
            "address_in_ocr",
            "total_in_ocr",
            "has_total_anchor",
            "has_date_anchor",
            "has_cash_anchor",
            "all_fields_present",
            "any_field_missing",
            "ocr_is_empty",
        ]
    ]
    .mean()
    .to_frame("fraction_true")
)

,fraction_true
company_present,0.939607
date_present,0.939607
address_present,0.938202
total_present,0.938202
company_in_ocr,0.917135
date_in_ocr,0.932584
address_in_ocr,0.880618
total_in_ocr,0.935393
has_total_anchor,0.987360
has_date_anchor,0.542135


In [60]:
# build the proxy verification labels from the feature table
sroie_labels_df = sroie_features.sroie_proxy_label_dataframe(sroie_df)

display(sroie_labels_df.head())
print(sroie_labels_df.shape)

,doc_id,company_hard,date_hard,address_hard,total_hard,low_ocr_support,proxy_risk_score,proxy_verify,proxy_high_risk
0,X00016469612,True,False,False,False,False,1,False,False
1,X00016469619,False,False,False,False,False,0,False,False
2,X00016469620,True,False,False,True,False,2,True,False
3,X00016469622,False,False,False,False,False,0,False,False
4,X00016469623,False,False,False,True,False,1,False,False


(712, 9)


In [61]:
# inspect proxy-label balance before using these labels downstream
display(
    sroie_labels_df[
        [
            "company_hard",
            "date_hard",
            "address_hard",
            "total_hard",
            "low_ocr_support",
            "proxy_verify",
            "proxy_high_risk",
        ]
    ]
    .mean()
    .to_frame("fraction_true")
)

display(sroie_labels_df["proxy_risk_score"].value_counts().sort_index())

,fraction_true
company_hard,0.080056
date_hard,0.067416
address_hard,0.116573
total_hard,0.122191
low_ocr_support,0.012640
proxy_verify,0.074438
proxy_high_risk,0.063202


proxy_risk_score
0    575
1     84
2      8
3      2
4     37
5      6
Name: count, dtype: int64

In [62]:
# rebuild proxy labels after revising the rules and inspect balance again
sroie_labels_df = sroie_features.sroie_proxy_label_dataframe(sroie_df)

display(
    sroie_labels_df[
        [
            "company_hard",
            "date_hard",
            "address_hard",
            "total_hard",
            "low_ocr_support",
            "proxy_verify",
            "proxy_high_risk",
        ]
    ]
    .mean()
    .to_frame("fraction_true")
)

display(sroie_labels_df["proxy_risk_score"].value_counts().sort_index())

,fraction_true
company_hard,0.080056
date_hard,0.067416
address_hard,0.116573
total_hard,0.122191
low_ocr_support,0.012640
proxy_verify,0.074438
proxy_high_risk,0.063202


proxy_risk_score
0    575
1     84
2      8
3      2
4     37
5      6
Name: count, dtype: int64

In [63]:
# combine features and proxy labels into one final SROIE modeling table
sroie_model_df = sroie_df.merge(sroie_labels_df, on="doc_id", how="left")

display(sroie_model_df.head())
print(sroie_model_df.shape)

,doc_id,dataset,split,img_width,img_height,n_tokens,n_boxes,ocr_char_count,ocr_word_count,company_present,...,ocr_is_empty,aspect_ratio,company_hard,date_hard,address_hard,total_hard,low_ocr_support,proxy_risk_score,proxy_verify,proxy_high_risk
0,X00016469612,SROIE,train,463.0,1013.0,44,44,441,84,True,...,False,0.457058,True,False,False,False,False,1,False,False
1,X00016469619,SROIE,train,439.0,1004.0,48,48,637,102,True,...,False,0.437251,False,False,False,False,False,0,False,False
2,X00016469620,SROIE,train,459.0,949.0,54,54,668,121,True,...,False,0.483667,True,False,False,True,False,2,True,False
3,X00016469622,SROIE,train,461.0,933.0,60,60,525,105,True,...,False,0.494105,False,False,False,False,False,0,False,False
4,X00016469623,SROIE,train,463.0,1026.0,61,61,735,142,True,...,False,0.451267,False,False,False,True,False,1,False,False


(712, 46)


In [64]:
# inspect a few high-risk receipts to make sure the proxy labels are not obviously broken
high_risk_ids = (
    sroie_model_df
    .sort_values(["proxy_high_risk", "proxy_risk_score"], ascending=[False, False])["doc_id"]
    .head(5)
    .tolist()
)

for doc_id in high_risk_ids:
    record = next(r for r in records if r.doc_id == doc_id)
    print("=" * 80)
    print(doc_id)
    print(record.fields)
    print(record.ocr_tokens[:25])

display(
    sroie_model_df[
        [
            "doc_id",
            "proxy_risk_score",
            "proxy_verify",
            "proxy_high_risk",
            "company_in_ocr",
            "date_in_ocr",
            "address_in_ocr",
            "total_in_ocr",
            "exact_total_matches",
            "n_amount_like_tokens",
            "n_date_like_tokens",
            "ocr_is_empty",
        ]
    ]
    .sort_values(["proxy_high_risk", "proxy_risk_score"], ascending=[False, False])
    .head(10)
)

X51005433492(1)
{}
[]
X51005442384(1)
{}
[]
X51005605333(1)
{}
[]
X51005685355(2)
{}
[]
X51005685357(2)
{}
[]


,doc_id,proxy_risk_score,proxy_verify,proxy_high_risk,company_in_ocr,date_in_ocr,address_in_ocr,total_in_ocr,exact_total_matches,n_amount_like_tokens,n_date_like_tokens,ocr_is_empty
34,X51005433492(1),5,True,True,False,False,False,False,0,0,0,True
64,X51005442384(1),5,True,True,False,False,False,False,0,0,0,True
119,X51005605333(1),5,True,True,False,False,False,False,0,0,0,True
169,X51005685355(2),5,True,True,False,False,False,False,0,0,0,True
172,X51005685357(2),5,True,True,False,False,False,False,0,0,0,True
566,X51007339118(1),5,True,True,False,False,False,False,0,0,0,True
56,X51005442376(1),4,True,True,False,False,False,False,0,4,0,False
58,X51005442378(1),4,True,True,False,False,False,False,0,9,0,False
60,X51005442379(1),4,True,True,False,False,False,False,0,4,0,False
62,X51005442383(1),4,True,True,False,False,False,False,0,17,0,False


In [65]:
# save the completed SROIE feature-plus-label table for later baseline and modeling notebooks
output_path = PROJECT_ROOT / "outputs" / "sroie_model_df.csv"

sroie_model_df.to_csv(output_path, index=False)
print(output_path)

/Users/quentin/Documents/Computer/School/CMU/Spring_2026/38616/Gated-Risk-MCP/outputs/sroie_model_df.csv
